# Machine Learning Zoomcamp 2026
## Homework 6: Decision Trees and Ensemble Learning

En esta tarea entrenamos y evaluamos modelos de árboles de decisión, Random Forest y XGBoost para predecir el rendimiento de combustible de vehículos (`fuel_efficiency_mpg`).

### 1. Carga y preparación del dataset

* Rellenar valores faltantes con 0.
* División de datos: 60% train, 20% val, 20% test con `random_state=1`.
* Vectorización con `DictVectorizer(sparse=True)`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Localizar dataset
csv_path = Path("../../cohorts/2026/data/car_fuel_efficiency_2026.csv").resolve()
if not csv_path.exists():
    csv_path = Path("cohorts/2026/data/car_fuel_efficiency_2026.csv").resolve()
if not csv_path.exists():
    csv_path = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(csv_path)
df = df.fillna(0)
print(f"Filas cargadas: {len(df)}")

In [ ]:
# División 60% / 20% / 20%
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop("fuel_efficiency_mpg").values
y_val = df_val.pop("fuel_efficiency_mpg").values
y_test = df_test.pop("fuel_efficiency_mpg").values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))
X_test = dv.transform(df_test.to_dict(orient="records"))

print(f"Dimensiones X_train: {X_train.shape}, X_val: {X_val.shape}")

---
### Question 1: Original splitting feature

Entrenamos un `DecisionTreeRegressor` con `max_depth=1` y determinamos qué variable original fue seleccionada para la partición inicial.

In [ ]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

split_feature_raw = dv.feature_names_[dt.tree_.feature[0]]
split_feature_original = split_feature_raw.split("=")[0]

print(f"Feature raw usada en el split: {split_feature_raw}")
print(f"Feature original: {split_feature_original}")

---
### Question 2: RMSE on validation

Entrenamos un `RandomForestRegressor` con `n_estimators=10`, `random_state=1`, `n_jobs=-1` y calculamos el RMSE en validación.

In [ ]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse_q2 = root_mean_squared_error(y_val, y_pred)
print(f"RMSE en validación (n_estimators=10): {rmse_q2:.4f}")

---
### Question 3: Number of estimators

Evaluamos diferentes valores de `n_estimators`: `[10, 50, 100, 150]` con `random_state=1` para encontrar cuál produce el menor RMSE.

In [ ]:
scores_n = {}
for n in [10, 50, 100, 150]:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    pred = rf_n.predict(X_val)
    rmse = root_mean_squared_error(y_val, pred)
    scores_n[n] = rmse
    print(f"n_estimators = {n:3d} -> RMSE: {rmse:.4f}")

best_n = min(scores_n, key=scores_n.get)
print(f"\nMejor n_estimators: {best_n} (RMSE: {scores_n[best_n]:.4f})")

---
### Question 4: Best max depth

Probamos combinaciones de `max_depth` en `[10, 15, 20, 25]` y `n_estimators` en `[10, 50, 100, 150]`, calculando el RMSE promedio para cada profundidad.

In [ ]:
depth_means = {}
for depth in [10, 15, 20, 25]:
    rmse_list = []
    for n in [10, 50, 100, 150]:
        rf_d = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf_d.fit(X_train, y_train)
        pred = rf_d.predict(X_val)
        rmse_list.append(root_mean_squared_error(y_val, pred))
    mean_rmse = np.mean(rmse_list)
    depth_means[depth] = mean_rmse
    print(f"max_depth = {depth:2d} -> Mean RMSE: {mean_rmse:.4f}")

best_depth = min(depth_means, key=depth_means.get)
print(f"\nMejor max_depth: {best_depth} (Mean RMSE: {depth_means[best_depth]:.4f})")

---
### Question 5: Most important feature

Entrenamos con `n_estimators=10`, `max_depth=20`, `random_state=1` y extraemos `feature_importances_` para las 4 candidatas:
- `vehicle_weight`
- `horsepower`
- `acceleration`
- `engine_displacement`

In [ ]:
rf_imp = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_imp.fit(X_train, y_train)

importances = dict(zip(dv.feature_names_, rf_imp.feature_importances_))
candidates = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]

print("Importancia de variables:")
for c in candidates:
    print(f"  {c:20s}: {importances.get(c, 0):.4f}")

most_important = max(candidates, key=lambda c: importances.get(c, 0))
print(f"\nVariable más importante: {most_important}")

---
### Question 6: XGBoost eta

Entrenamos XGBoost durante 100 rondas comparando `eta=0.3` frente a `eta=0.1`.

In [ ]:
features = dv.get_feature_names_out()
features_clean = [f.replace("=", "_").replace(" ", "_").replace("<", "_") for f in features]

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features_clean)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features_clean)
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_scores = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 0,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    pred_xgb = model.predict(dval)
    rmse = root_mean_squared_error(y_val, pred_xgb)
    xgb_scores[eta] = rmse
    print(f"eta = {eta} -> Validation RMSE: {rmse:.4f}")

best_eta = min(xgb_scores, key=xgb_scores.get)
print(f"\nMejor eta: {best_eta} (RMSE: {xgb_scores[best_eta]:.4f})")